In [ ]:
import json
import time
from pathlib import Path
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, lit, monotonically_increasing_id, row_number
from pyspark.sql.window import Window
from mxdataspark.core.storage.datalake import DataLakeManager
from mxdataspark.target.type.dynamics import Dynamics
from mxdataspark.target.type.dynamics.data_client import DataClient


In [ ]:
dbutils.widgets.text("dropbox_folder", "")
dbutils.widgets.text("config_name", "")
dbutils.widgets.text("secret_name", "")
dbutils.widgets.text("quarantine_download_folder", "")
dbutils.widgets.text("catalog_name", "mxdata_de1_dev")


In [ ]:
dropbox_folder = Path(dbutils.widgets.get("dropbox_folder"))
config_name = dbutils.widgets.get("config_name")
secret_name = dbutils.widgets.get("secret_name")
quarantine_download_folder = Path(dbutils.widgets.get("quarantine_download_folder"))
catalog_name = dbutils.widgets.get("catalog_name")


In [ ]:
def get_secret(secret_name) -> dict:
    secret = dbutils.secrets.get("primary-key-vault-scope", secret_name)
    return json.loads(secret)


def get_rtk(erp) -> str:
    return dbutils.secrets.get("primary-key-vault-scope", f"jdbc-{erp}-rtk")


def get_ddu_client(secret_name) -> Dynamics:
    secret = get_secret(secret_name)
    erp = secret["dynamics_erp"]
    rtk = get_rtk(erp)

    print(f"Creating DDU client for -> {erp}")
    client = DataClient.from_parameters_with_erp(
        erp,
        secret["dynamics_url"],
        secret["azure_tenant"],
        secret["client_id"],
        secret["client_secret"],
        rtk,
    )
    return Dynamics(client)


def read_from_excel(path) -> DataFrame:
    df = (
        spark.read.format("com.crealytics.spark.excel")
        .option("maxByteArraySize", 2147483647)
        .option("header", "true")
        .load(str(path))
    )
    return df


def load_file(ddu_client, config) -> None:
    file_name = config.get("file_name")
    target_entity = config.get("target_entity")
    legal_entity = config.get("legal_entity", None)
    key_columns = config.get("key_columns", "")
    mode = config.get("mode", "upsert")

    path = dropbox_folder / file_name
    source_file = path.stem
    df = read_from_excel(path)

    if legal_entity:
        df = df.withColumn("dataAreaId", lit(legal_entity.lower()))
        if isinstance(key_columns, str):
            key_columns = [k.strip() for k in key_columns.split(",")]
        if "dataareaid" not in {k.lower() for k in key_columns or []}:
            key_columns.append("dataAreaId")

    print(
        f"---> Loading file:{file_name} target_entity:{target_entity} legal_entity:{legal_entity} key_columns:{key_columns} mode:{mode}"
    )
    ddu_client.load_dataframe(df, target_entity, key_columns, source_file, mode)

    # Download quarantine
    print(f"     Downloading quarantine for {source_file}  run_id: {ddu_client.run_id}")
    quarantine_format = "csv"
    ddu_client.quarantine.export_quarantine(
        source_file,
        str(quarantine_download_folder / source_file) + f".{quarantine_format}",
        format=quarantine_format,
        run_id=ddu_client.run_id,
    )

    # Move file to archive
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    archive_path = str(path.parent / "archive" / f"{timestamp}_{file_name}")
    dbutils.fs.mv(str(path), archive_path)
    print(f"     File moved from {path} to {archive_path}")


def main() -> None:
    spark.catalog.setCurrentCatalog(catalog_name)

    config_path = dropbox_folder / config_name
    configs_df = read_from_excel(config_path)
    window = Window.partitionBy(lit(1)).orderBy(monotonically_increasing_id())
    configs_df = configs_df.withColumn("priority", row_number().over(window))

    files_to_process = DataLakeManager().get_files_from_path(
        path=str(dropbox_folder), exclude=["archive/", "quarantine/", config_name]
    )
    files_to_process = [
        str(Path(f).stem + Path(f).suffix)
        for f in files_to_process
        if f.endswith(".xlsx") and (Path(f).stem + Path(f).suffix) != config_name
    ]

    if len(files_to_process) == 0:
        print("No files to process")
        return

    files_to_process_df = spark.createDataFrame(
        [(f,) for f in files_to_process], ["file_name"]
    )

    # join the files to process with the config file
    df = files_to_process_df.join(configs_df, ["file_name"], "left")

    missing_target_entity = df.filter(col("target_entity").isNull()).select("file_name")
    if missing_target_entity.count() > 0:
        raise ValueError(
            f"Config is missing for these file(s): {missing_target_entity.rdd.map(lambda row: row[0]).collect()}"
        )

    ddu_client = get_ddu_client(secret_name)
    configs = (
        df.filter(col("target_entity").isNotNull())
        .rdd.sortBy(lambda row: row["priority"])
        .map(lambda row: row.asDict())
        .collect()
    )
    print(configs)
    [load_file(ddu_client, config) for config in configs]


In [ ]:
# Start the script
main()
